In [67]:
import pandas as pd
import numpy as np
import nltk
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

# === Load Data ===
df = pd.read_excel("labelling_manual.xlsx")

df['content'] = df['content'].astype(str)

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # Cleaning dasar
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    text = text.lower()

    # Tokenisasi
    tokens = word_tokenize(text)

    # Stemming → Lemmatization (dua-duanya dikerjakan)
    tokens = [lemmatizer.lemmatize(stemmer.stem(w)) for w in tokens]

    return " ".join(tokens)

df['clean'] = df['content'].apply(preprocess)

X_text = df['clean']
y = df['sentiment']

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


TF IDF + LOGISTIC REGRESSION

In [68]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# === Vectorization TF-IDF ===
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(X_text)

# === Train-test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

# === Model Logistic Regression ===
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

# === Prediksi ===
preds = model.predict(X_test)

# === Akurasi ===
acc = accuracy_score(y_test, preds)
print("TF-IDF + LR:", acc)

TF-IDF + LR: 0.72


BOW + NAIVE BAYES

In [69]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# === Vectorization BoW ===
bow = CountVectorizer()
X = bow.fit_transform(X_text)

# === Train-test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

# === Model Naive Bayes ===
model = MultinomialNB()
model.fit(X_train, y_train)

# === Prediksi ===
preds = model.predict(X_test)

# === Akurasi ===
acc = accuracy_score(y_test, preds)
print("BoW + Naive Bayes:", acc)


BoW + Naive Bayes: 0.74


WORD2VEC + SVM

In [70]:
pip install gensim

In [71]:
from gensim.models import Word2Vec
from sklearn.svm import SVC

# === Word2Vec Training ===
tokens = [text.split() for text in X_text]
w2v = Word2Vec(tokens, vector_size=100, window=5, min_count=1)

# === Convert setiap kalimat ke vektor rata-rata ===
def vectorize(words):
    return np.mean([w2v.wv[w] for w in words if w in w2v.wv], axis=0)

X = np.array([vectorize(sentence.split()) for sentence in X_text])

# === Train-test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=3
)

# === Model SVM ===
model = SVC(probability=True)
model.fit(X_train, y_train)

# === Prediksi ===
preds = model.predict(X_test)

# === Akurasi ===
acc = accuracy_score(y_test, preds)
print("Word2Vec + SVM:", acc)


Word2Vec + SVM: 0.58


PERCOBAAN 4: GLOVE + LOGISTIC RERESSION

In [72]:
from sklearn.linear_model import LogisticRegression

# === Fake GloVe embedding (random vectors untuk setiap kata) ===

tokens = [t.split() for t in X_text]
vocab = {w for sent in tokens for w in sent}

word_to_vec = {w: np.random.rand(100) for w in vocab}

def vectorize_glove(words):
    return np.mean([word_to_vec[w] for w in words if w in word_to_vec], axis=0)

X = np.array([vectorize_glove(text.split()) for text in X_text])

# === Train-test Split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=4
)

# === Model Logistic Regression sebagai classifier ===
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

# === Prediksi ===
preds = model.predict(X_test)

# === Akurasi ===
acc = accuracy_score(y_test, preds)
print("GloVe + Logistic Regression:", acc)


GloVe + Logistic Regression: 0.76


TF IDF + SVM

In [73]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

# === TF-IDF Training ===
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(X_text)   # X_text = data teks kamu

# === Train-test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=3
)

# === Model SVM ===
model = SVC(kernel="linear", probability=True)
model.fit(X_train, y_train)

# === Prediksi ===
preds = model.predict(X_test)

# === Akurasi ===
acc = accuracy_score(y_test, preds)
print("TF-IDF + SVM:", acc)


TF-IDF + SVM: 0.76


WORD2VEC + SVM

In [74]:
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import numpy as np

# === Tokenisasi dasar ===
tokens = [text.split() for text in X_text]   # X_text = kolom teks bersih
y = y                                        # label sentiment

# === Train Word2Vec ===
w2v = Word2Vec(
    sentences=tokens,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

# === Fungsi vectorize: rata-rata embedding per kalimat ===
def vectorize(words):
    vecs = [w2v.wv[w] for w in words if w in w2v.wv]
    if len(vecs) == 0:
        return np.zeros(100)
    return np.mean(vecs, axis=0)

# === Konversi seluruh teks menjadi embedding ===
X = np.array([vectorize(sent) for sent in tokens])

# === Train-test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=3
)

# === SVM classifier ===
model = SVC(kernel='linear', probability=True)
model.fit(X_train, y_train)

# === Prediksi ===
preds = model.predict(X_test)

# === Akurasi ===
acc = accuracy_score(y_test, preds)
print("Word2Vec + SVM:", acc)


Word2Vec + SVM: 0.4


TF IDF + NAIVES BAYES

In [75]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# === TF-IDF ===
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(X_text)   # X_text = list / kolom teks
y = y                                        # label sentiment

# === Train-test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=3
)

# === Model Naive Bayes ===
model = MultinomialNB()
model.fit(X_train, y_train)

# === Prediksi ===
preds = model.predict(X_test)

# === Akurasi ===
acc = accuracy_score(y_test, preds)
print("TF-IDF + Naive Bayes:", acc)


TF-IDF + Naive Bayes: 0.76
